# Structure probing in code LLMs — CoST (Java + Python)

Layer-by-layer linear probes that ask: **where in `Qwen/Qwen2.5-1.5B` is a code token's
*syntactic class* (identifier / keyword / operator / string / number / punctuation /
bool-null) linearly decodable?**

Two experiments over the parallel **CoST** corpus (`consolidated_data.csv`, GeeksforGeeks
problems aligned across languages):

1. **Per-language** layerwise curves for Python and Java.
2. **Cross-lingual transfer** — train the probe on Python activations, test on Java (and
   the reverse) at the same layer. Because CoST pairs the *same* algorithm in both
   languages, this tests whether Qwen encodes code structure in a shared, language-agnostic
   subspace.

We run the **same pipeline on two models** — `Qwen/Qwen2.5-1.5B` (base) and
`Qwen/Qwen2.5-Coder-1.5B` (code-pretrained) — so every result is directly comparable.
Each model is loaded, probed, and freed before the next, so host RAM stays flat.

Beyond the two core experiments we add three rigor extensions, run for **both** models:

3. **Hewitt control task** — assign each *token type* a random fixed class and re-probe.
   `selectivity = real macro-F1 − control macro-F1` separates "the representation encodes
   structure" from "the probe just memorizes token identity".
4. **Finer identifier-role probe** — split `identifier` into `variable / function / type /
   parameter` (parent-context labels). Token identity is no longer enough here, so this is
   the harder, more revealing probe.
5. **Per-class layerwise curves** — which structural class peaks at which depth.

Runs on a Colab **T4** (no GPU training — only forward passes; the probes are CPU
`LogisticRegression`). With both models + all extensions expect ~30–45 min. An A100/L4 is
faster but not required. See `docs/colab_pro_setup.md`.

## 1. Setup

In [ ]:
import subprocess, sys

def _ensure(pkgs):
    missing = []
    for mod, pip_name in pkgs:
        try:
            __import__(mod)
        except ModuleNotFoundError:
            missing.append(pip_name)
    if missing:
        print('Installing:', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

_ensure([
    ('torch', 'torch'),
    ('transformers', 'transformers'),
    ('sklearn', 'scikit-learn'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('tqdm', 'tqdm'),
    ('tree_sitter', 'tree-sitter>=0.23'),
    ('tree_sitter_python', 'tree-sitter-python>=0.23'),
    ('tree_sitter_java', 'tree-sitter-java>=0.23'),
])

In [ ]:
import os, json, csv, random
from collections import Counter, defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

import warnings; warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- Experiment configuration (tune for speed vs coverage) -------------------
# Both models share the Qwen2.5 tokenizer, so token sites are identical across them.
MODELS               = ['Qwen/Qwen2.5-1.5B', 'Qwen/Qwen2.5-Coder-1.5B']
MODEL_NAME           = MODELS[0]   # used only for the standalone labeling demos below
RUN_CONTROL_TASK     = True    # Hewitt selectivity: random fixed class per token type
RUN_IDENTIFIER_ROLES = True    # finer probe: variable / function / type / parameter
MAX_SEQ_LEN          = 512     # tokenizer truncation
MAX_PROGRAMS_PER_LANG = 300    # cap programs per language (CoST has ~1417 aligned)
MAX_TOKENS_PER_LANG  = 12000   # subsample budget -> bounds host RAM + probe time
INCLUDE_COMMENTS     = False   # 7-class structural space (drop comment tokens)
TEST_FRACTION        = 0.2     # problem-level split
LAYER_STRIDE         = 1       # probe every Nth layer (set 2 or 4 for a fast first pass)

OUTPUT_DIR = os.path.join(os.getcwd(), 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

## 2. Model loader (hidden states only)

`load_model` is called once per model inside the driver loop. Here we only build a
tokenizer so the labeling demos below can run (Qwen2.5 and Qwen2.5-Coder share one vocab).

In [ ]:
def load_model(name):
    """Load tokenizer + base AutoModel (hidden states only) onto DEVICE."""
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    assert tok.is_fast, 'Need a fast tokenizer for offset_mapping alignment.'
    dtype = torch.float16 if DEVICE in ('cuda', 'mps') else torch.float32
    mdl = AutoModel.from_pretrained(
        name, output_hidden_states=True, trust_remote_code=True, torch_dtype=dtype,
    )
    mdl.eval().to(DEVICE)
    return tok, mdl

# Tokenizer for the standalone labeling demos below. The per-model driver loop rebinds
# the globals `tokenizer`, `model`, `NUM_LAYERS`, `HIDDEN_SIZE`, `N_HS`, `PROBE_LAYERS`.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Demo tokenizer ready:', MODEL_NAME)
print('Models to run:', MODELS)

## 3. Load CoST (Java + Python aligned)

Upload `consolidated_data.csv` via the **Files** panel (left), or mount Drive and point
`CSV_PATH` at it. We keep only problems that have **both** a Java and a Python program so
the two languages stay aligned on `problem_id`.

In [ ]:
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

# Try common locations; override CSV_PATH if needed.
_CANDIDATES = ['consolidated_data.csv', '/content/consolidated_data.csv',
               '/content/drive/MyDrive/consolidated_data.csv', '../consolidated_data.csv']
CSV_PATH = next((p for p in _CANDIDATES if os.path.isfile(p)), 'consolidated_data.csv')
print('CSV_PATH =', CSV_PATH)

LANG_COLUMNS = {'Java': 'java', 'Python': 'python'}

def load_cost(csv_path, max_programs_per_lang):
    """Return {lang: [(problem_id, code), ...]} for problems with BOTH languages."""
    out = {tag: [] for tag in LANG_COLUMNS.values()}
    with open(csv_path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pid = (row.get('Problem ID') or '').strip()
            codes = {tag: (row.get(col) or '').strip() for col, tag in LANG_COLUMNS.items()}
            if not pid or not all(codes.values()):
                continue
            if all(len(out[t]) >= max_programs_per_lang for t in codes):
                break
            for tag, code in codes.items():
                if len(out[tag]) < max_programs_per_lang:
                    out[tag].append((int(pid), code))
    return out

programs = load_cost(CSV_PATH, MAX_PROGRAMS_PER_LANG)
for lang, items in programs.items():
    print(f'{lang:8s} {len(items)} programs')

## 4. Structural labels via tree-sitter (shared Java/Python space)

Walk every **leaf** node and assign one coarse `structural_class`. Same logic as
`scripts/structure_labels.py`, inlined so the notebook is self-contained.

In [ ]:
import tree_sitter_python as tsp, tree_sitter_java as tsj
from tree_sitter import Language, Parser

PARSERS = {'python': Parser(Language(tsp.language())), 'java': Parser(Language(tsj.language()))}

# 7-class structural space (comment handled separately via INCLUDE_COMMENTS).
CLASSES = ['identifier', 'keyword', 'operator', 'string', 'number', 'punctuation', 'bool_null']
if INCLUDE_COMMENTS:
    CLASSES = CLASSES + ['comment']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

_IDENTIFIER_TYPES = {'identifier', 'type_identifier', 'field_identifier', 'scoped_identifier',
                     'scoped_type_identifier', 'dotted_name'}
_BOOL_NULL_TYPES = {'true', 'false', 'none', 'null', 'null_literal'}
_PUNCT = set('()[]{},;.') | {'->', '::', '...'}

def classify_leaf(node_type, text, is_named):
    t = node_type.lower()
    if t in _BOOL_NULL_TYPES or text in ('True', 'False', 'None', 'true', 'false', 'null'):
        return 'bool_null'
    if t in _IDENTIFIER_TYPES:
        return 'identifier'
    if 'comment' in t:
        return 'comment'
    if 'string' in t or 'char' in t or t == 'escape_sequence':
        return 'string'
    if any(k in t for k in ('integer', 'float', 'decimal', 'hex', 'number', 'literal')) \
            and (text[:1].isdigit() or text[:1] in '+-.'):
        return 'number'
    s = text.strip()
    if s and (s.isidentifier() or s.replace('_', 'a').isalpha()):
        return 'keyword'
    if s in _PUNCT or (len(s) == 1 and s in '()[]{},;.:'):
        return 'punctuation'
    return 'operator'

def byte_to_char_map(code):
    data = code.encode('utf-8'); m = [0] * (len(data) + 1)
    ci = bi = 0
    for ch in code:
        n = len(ch.encode('utf-8'))
        for b in range(bi, bi + n):
            m[b] = ci
        bi += n; ci += 1
    m[len(data)] = ci
    return data, m

def char_class_array(code, language):
    """Per-character class index (or -1) from tree-sitter leaves."""
    data, b2c = byte_to_char_map(code)
    tree = PARSERS[language].parse(data)
    arr = np.full(len(code), -1, dtype=np.int64)
    stack = [tree.root_node]; leaves = []
    while stack:
        node = stack.pop()
        if node.child_count == 0:
            leaves.append(node)
        else:
            stack.extend(reversed(node.children))
    for node in leaves:
        sb, eb = node.start_byte, node.end_byte
        if eb <= sb or node.is_error:
            continue
        text = data[sb:eb].decode('utf-8', errors='replace')
        cls = classify_leaf(node.type, text, node.is_named)
        if cls not in CLASS_TO_IDX:
            continue
        arr[b2c[sb]:b2c[eb]] = CLASS_TO_IDX[cls]
    return arr

# quick demo
_demo = "def f(x):\n    ok = True\n    if ok and x > 0:\n        return 1\n"
_ca = char_class_array(_demo, 'python')
print('demo char-classes (first 30):', _ca[:30].tolist())

In [ ]:
def token_labels(code, language, char_fn=char_class_array):
    """Tokenize once; return (input_ids, [(token_index, class_idx), ...]).
    Each token inherits the class of the first labeled character it covers.
    `char_fn` is the per-character labeler (structural class or identifier role)."""
    enc = tokenizer(code, truncation=True, max_length=MAX_SEQ_LEN,
                    return_offsets_mapping=True, return_tensors=None)
    offsets = enc['offset_mapping']
    input_ids = enc['input_ids']
    ca = char_fn(code, language)
    n = len(ca)
    labeled = []
    for ti, (cs, ce) in enumerate(offsets):
        if ce <= cs:
            continue  # special / empty
        lab = -1
        for pos in range(cs, min(ce, n)):
            if ca[pos] >= 0:
                lab = int(ca[pos]); break
        if lab >= 0:
            labeled.append((ti, lab))
    return input_ids, labeled

_ids, _lab = token_labels(_demo, 'python')
_toks = tokenizer.convert_ids_to_tokens(_ids)
print('Demo token -> class:')
for ti, lab in _lab[:20]:
    print(f'  {repr(_toks[ti]):16s} {CLASSES[lab]}')

In [ ]:
# ---- Finer identifier-role labels (parent-context based) ---------------------
# Splits the `identifier` class into variable / function / type / parameter. Token identity
# alone can't solve this, so it is a much harder probe than the coarse structural class.
ID_ROLES = ['variable', 'function', 'type', 'parameter']
ID_ROLE_TO_IDX = {r: i for i, r in enumerate(ID_ROLES)}

def _identifier_role(node, language):
    """Classify an identifier leaf by its surrounding syntax."""
    if 'type_identifier' in node.type or node.type == 'scoped_type_identifier':
        return 'type'
    p = node.parent
    if p is None:
        return 'variable'
    pt = p.type
    if language == 'python':
        if pt == 'function_definition' and p.child_by_field_name('name') == node:
            return 'function'
        if pt == 'class_definition' and p.child_by_field_name('name') == node:
            return 'type'
        if pt == 'call' and p.child_by_field_name('function') == node:
            return 'function'
        if pt == 'attribute':
            gp = p.parent
            if gp is not None and gp.type == 'call' \
               and gp.child_by_field_name('function') == p \
               and p.child_by_field_name('attribute') == node:
                return 'function'
        if pt in ('parameters', 'lambda_parameters', 'default_parameter',
                  'typed_parameter', 'typed_default_parameter'):
            return 'parameter'
        return 'variable'
    # java
    if pt == 'method_declaration' and p.child_by_field_name('name') == node:
        return 'function'
    if pt == 'method_invocation' and p.child_by_field_name('name') == node:
        return 'function'
    if pt in ('class_declaration', 'interface_declaration', 'enum_declaration') \
       and p.child_by_field_name('name') == node:
        return 'type'
    if pt in ('formal_parameter', 'spread_parameter', 'catch_formal_parameter'):
        return 'parameter'
    return 'variable'

def id_role_char_array(code, language):
    """Per-character identifier-role index (or -1 for non-identifier chars)."""
    data, b2c = byte_to_char_map(code)
    tree = PARSERS[language].parse(data)
    arr = np.full(len(code), -1, dtype=np.int64)
    stack = [tree.root_node]; leaves = []
    while stack:
        node = stack.pop()
        if node.child_count == 0:
            leaves.append(node)
        else:
            stack.extend(reversed(node.children))
    for node in leaves:
        sb, eb = node.start_byte, node.end_byte
        if eb <= sb or node.is_error:
            continue
        text = data[sb:eb].decode('utf-8', errors='replace')
        if classify_leaf(node.type, text, node.is_named) != 'identifier':
            continue
        arr[b2c[sb]:b2c[eb]] = ID_ROLE_TO_IDX[_identifier_role(node, language)]
    return arr

_ids2, _lab2 = token_labels(_demo, 'python', id_role_char_array)
_toks2 = tokenizer.convert_ids_to_tokens(_ids2)
print('Demo identifier roles:')
for ti, lab in _lab2[:20]:
    print(f'  {repr(_toks2[ti]):16s} {ID_ROLES[lab]}')

## 5. Build the probe site index + subsample

Structure labels are *dense* (every token), so we cap the number of probe tokens per
language with **per-class balancing** before extracting activations. This is what keeps the
activation tensors to a few GB (the GPU is never the bottleneck — host RAM is).

In [ ]:
def build_sites(programs_lang, language, char_fn=char_class_array):
    """For one language: tokenize every program and collect labeled token sites.
    `char_fn` selects the label space (structural class or identifier role)."""
    sites = []   # (prog_idx, token_index, class_idx)
    cache = []   # prog_idx -> (problem_id, input_ids)
    for prog_idx, (pid, code) in enumerate(programs_lang):
        try:
            input_ids, labeled = token_labels(code, language, char_fn)
        except Exception:
            cache.append((pid, []))
            continue
        cache.append((pid, input_ids))
        for ti, lab in labeled:
            sites.append((prog_idx, ti, lab))
    return sites, cache

def subsample_balanced(sites, max_tokens):
    """Cap total tokens, balancing per class as far as supply allows."""
    by_cls = defaultdict(list)
    for s in sites:
        by_cls[s[2]].append(s)
    n_cls = len(by_cls)
    per = max(1, max_tokens // max(1, n_cls))
    keep = []
    leftover_pool = []
    for cls, lst in by_cls.items():
        random.shuffle(lst)
        keep.extend(lst[:per])
        leftover_pool.extend(lst[per:])
    # fill remaining budget from leftovers (classes with extra supply)
    random.shuffle(leftover_pool)
    room = max_tokens - len(keep)
    if room > 0:
        keep.extend(leftover_pool[:room])
    random.shuffle(keep)
    return keep

def build_site_index(char_fn=char_class_array, label_names=None):
    """Build + subsample token sites for both languages with the given label space.
    Returns {lang: {'sites': [...], 'cache': [...]}}. Uses the current global tokenizer."""
    idx = {}
    for lang in ['python', 'java']:
        sites, cache = build_sites(programs[lang], lang, char_fn)
        kept = subsample_balanced(sites, MAX_TOKENS_PER_LANG)
        idx[lang] = {'sites': kept, 'cache': cache}
        if label_names is not None:
            dist = Counter(c for _, _, c in kept)
            print(f'  {lang:8s} {len(sites):6d} labeled -> kept {len(kept):6d}   ' +
                  '  '.join(f'{label_names[c][:4]}:{dist.get(c, 0)}'
                            for c in range(len(label_names))))
    return idx

## 6. Extract per-layer hidden states

One forward per program (only programs with kept sites). For each kept token we store its
residual vector at every layer `0..NUM_LAYERS`.

In [ ]:
@torch.no_grad()
def extract_activations(site_d):
    """Forward each program once; store residual vectors at every layer for kept tokens.
    Returns {'X': [N_HS, n, H] float16, 'y', 'groups' (problem_id), 'tokids' (input id)}."""
    sites = site_d['sites']
    cache = site_d['cache']
    by_prog = defaultdict(list)
    for prog_idx, ti, lab in sites:
        by_prog[prog_idx].append((ti, lab))
    n = len(sites)
    # float16 halves host RAM: N_HS x n x H x 2 bytes (~1 GB/lang at 12k tokens).
    X = np.zeros((N_HS, n, HIDDEN_SIZE), dtype=np.float16)
    y = np.zeros(n, dtype=np.int64)
    groups = np.zeros(n, dtype=np.int64)   # problem_id, for leakage-free split
    tokids = np.zeros(n, dtype=np.int64)   # token id, for the Hewitt control task
    w = 0
    for prog_idx, toks in tqdm(by_prog.items(), desc='forwards', leave=False):
        pid, input_ids = cache[prog_idx]
        if not input_ids:
            continue
        ids = torch.tensor([input_ids], device=DEVICE)
        out = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)
        hs = out.hidden_states  # tuple len N_HS, each [1, seq, H]
        seq = ids.shape[1]
        stack = np.stack([hs[L][0].float().cpu().numpy() for L in range(N_HS)], axis=0)  # [N_HS, seq, H]
        for ti, lab in toks:
            if ti >= seq:
                continue
            X[:, w, :] = stack[:, ti, :]
            y[w] = lab
            groups[w] = pid
            tokids[w] = input_ids[ti]
            w += 1
        del out, hs, stack
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    return {'X': X[:, :w, :], 'y': y[:w], 'groups': groups[:w], 'tokids': tokids[:w]}

## 7. Probe training — per-language, layer by layer

Multinomial `LogisticRegression` at each layer. Split is by **`problem_id`** so no program
straddles train/test. Metric: macro-F1 (robust to the identifier/punctuation imbalance).

In [ ]:
def split_by_group(groups, test_fraction=TEST_FRACTION, seed=SEED):
    uniq = np.unique(groups)
    rng = np.random.RandomState(seed); rng.shuffle(uniq)
    n_test = max(1, int(len(uniq) * test_fraction))
    test_groups = set(uniq[:n_test].tolist())
    test_mask = np.array([g in test_groups for g in groups])
    return ~test_mask, test_mask

def fit_probe(X_tr, y_tr):
    # StandardScaler makes lbfgs converge ~5-10x faster on raw activations.
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=200, class_weight='balanced', C=1.0, solver='lbfgs'),
    )
    clf.fit(X_tr, y_tr)
    return clf

def probe_layers_within(act, n_classes, y_override=None, keep_probes=True):
    """Per-layer probe on one activation set. `y_override` runs the Hewitt control task
    (random fixed class per token type) on the same train/test split. Tracks per-class F1."""
    X, g = act['X'], act['groups']
    y = act['y'] if y_override is None else y_override
    tr, te = split_by_group(g)
    maj = Counter(y[tr].tolist()).most_common(1)[0][0]
    baseline = f1_score(y[te], np.full(y[te].shape, maj), average='macro',
                        labels=range(n_classes), zero_division=0)
    macro = [np.nan] * N_HS; acc = [np.nan] * N_HS
    per_class = [[np.nan] * N_HS for _ in range(n_classes)]
    res = {'macro_f1': macro, 'acc': acc, 'baseline_f1': baseline,
           'per_class_f1': per_class, 'test_idx': te, 'probes': {}}
    for L in tqdm(PROBE_LAYERS, desc='within', leave=False):
        clf = fit_probe(X[L][tr], y[tr])
        yp = clf.predict(X[L][te])
        macro[L] = f1_score(y[te], yp, average='macro', labels=range(n_classes), zero_division=0)
        acc[L] = accuracy_score(y[te], yp)
        pcf = f1_score(y[te], yp, average=None, labels=range(n_classes), zero_division=0)
        for c in range(n_classes):
            per_class[c][L] = pcf[c]
        if keep_probes:
            res['probes'][L] = clf
    return res

def make_control_labels(act, n_classes, seed=SEED):
    """Hewitt control: assign each token *type* (input id) a fixed random class,
    sampled from the empirical class marginal so the control task has matched priors."""
    rng = np.random.RandomState(seed)
    counts = np.bincount(act['y'], minlength=n_classes).astype(float)
    probs = counts / counts.sum()
    cmap = {int(t): int(rng.choice(n_classes, p=probs)) for t in np.unique(act['tokids'])}
    return np.array([cmap[int(t)] for t in act['tokids']], dtype=np.int64)

## 8. Cross-lingual transfer

Train on **all** of one language's tokens, test on the other — same layer index. High
transfer means the structural code lives in a shared subspace.

In [ ]:
def probe_layers_cross(src_act, dst_act, n_classes):
    """Train probe on all of src, test on all of dst, at each layer index."""
    Xs, ys = src_act['X'], src_act['y']
    Xd, yd = dst_act['X'], dst_act['y']
    macro = [np.nan] * N_HS; acc = [np.nan] * N_HS
    for L in tqdm(PROBE_LAYERS, desc='cross', leave=False):
        clf = fit_probe(Xs[L], ys)
        yp = clf.predict(Xd[L])
        macro[L] = f1_score(yd, yp, average='macro', labels=range(n_classes), zero_division=0)
        acc[L] = accuracy_score(yd, yp)
    return {'macro_f1': macro, 'acc': acc}

## 9. Run both models

For each model: load → build sites → extract activations → run within-language,
cross-lingual, control-task, and identifier-role probes → store numeric results → free the
model and activations. Only **one** model's activations live in RAM at a time. On a T4 this
is ~15–20 min per model.

In [ ]:
import gc

ALL = {}
for name in MODELS:
    print('\n' + '=' * 64 + f'\nMODEL: {name}\n' + '=' * 64)
    tokenizer, model = load_model(name)
    NUM_LAYERS = model.config.num_hidden_layers
    HIDDEN_SIZE = model.config.hidden_size
    N_HS = NUM_LAYERS + 1
    PROBE_LAYERS = sorted(set(list(range(0, N_HS, LAYER_STRIDE)) + [N_HS - 1]))
    print(f'  layers={NUM_LAYERS}  hidden={HIDDEN_SIZE}  probing {len(PROBE_LAYERS)} layers')

    # --- Structural-class pipeline ---
    print('Structural-class sites:')
    site_index = build_site_index(char_class_array, CLASSES)
    acts = {lang: extract_activations(site_index[lang]) for lang in ['python', 'java']}
    nC = len(CLASSES)
    within = {lang: probe_layers_within(acts[lang], nC) for lang in ['python', 'java']}
    cross = {'python->java': probe_layers_cross(acts['python'], acts['java'], nC),
             'java->python': probe_layers_cross(acts['java'], acts['python'], nC)}
    for lang in within:
        b = int(np.nanargmax(within[lang]['macro_f1']))
        print(f'  {lang}: best L{b} macro-F1={within[lang]["macro_f1"][b]:.3f} '
              f'(baseline {within[lang]["baseline_f1"]:.3f})')

    # --- Hewitt control task ---
    control = {}
    if RUN_CONTROL_TASK:
        for lang in ['python', 'java']:
            yc = make_control_labels(acts[lang], nC)
            control[lang] = probe_layers_within(acts[lang], nC, y_override=yc, keep_probes=False)
        b = int(np.nanargmax(within['python']['macro_f1']))
        print(f'  control: Py selectivity @L{b} = '
              f'{within["python"]["macro_f1"][b] - control["python"]["macro_f1"][b]:.3f}')

    # --- Confusion data (Python structural, best layer) — kept before freeing acts ---
    bl = int(np.nanargmax(within['python']['macro_f1']))
    te = within['python']['test_idx']
    confusion = {'best_layer': bl,
                 'y_true': acts['python']['y'][te].copy(),
                 'y_pred': within['python']['probes'][bl].predict(acts['python']['X'][bl][te])}

    del acts
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    # --- Finer identifier-role pipeline ---
    idrole = {}
    if RUN_IDENTIFIER_ROLES:
        print('Identifier-role sites:')
        id_index = build_site_index(id_role_char_array, ID_ROLES)
        id_acts = {lang: extract_activations(id_index[lang]) for lang in ['python', 'java']}
        nR = len(ID_ROLES)
        idrole['within'] = {lang: probe_layers_within(id_acts[lang], nR, keep_probes=False)
                            for lang in ['python', 'java']}
        idrole['cross'] = {'python->java': probe_layers_cross(id_acts['python'], id_acts['java'], nR),
                           'java->python': probe_layers_cross(id_acts['java'], id_acts['python'], nR)}
        for lang in idrole['within']:
            b = int(np.nanargmax(idrole['within'][lang]['macro_f1']))
            print(f'  id-role {lang}: best L{b} macro-F1='
                  f'{idrole["within"][lang]["macro_f1"][b]:.3f} '
                  f'(baseline {idrole["within"][lang]["baseline_f1"]:.3f})')
        del id_acts
        gc.collect()

    ALL[name] = {'within': within, 'cross': cross, 'control': control, 'idrole': idrole,
                 'confusion': confusion, 'N_HS': N_HS, 'NUM_LAYERS': NUM_LAYERS}

    del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print('\nDone. Models:', list(ALL.keys()))

## 10. Visualization — base vs Coder

In [ ]:
# Structural-class probe vs layer — one panel per model (within + cross-lingual).
names = list(ALL.keys())
fig, axes = plt.subplots(1, len(names), figsize=(7 * len(names), 5.5), squeeze=False)
for ax, name in zip(axes[0], names):
    R = ALL[name]; layers = list(range(R['N_HS']))
    w, c = R['within'], R['cross']
    ax.plot(layers, w['python']['macro_f1'], '-o', ms=3, label='Python (within)', color='#1f77b4')
    ax.plot(layers, w['java']['macro_f1'],   '-s', ms=3, label='Java (within)',   color='#ff7f0e')
    ax.plot(layers, c['python->java']['macro_f1'], '--^', ms=3, label='Train Py → Test Java', color='#2ca02c')
    ax.plot(layers, c['java->python']['macro_f1'], '--v', ms=3, label='Train Java → Test Py', color='#d62728')
    ax.axhline(w['python']['baseline_f1'], color='gray', ls=':', alpha=0.7, label='Majority baseline')
    ax.set_title(name.split('/')[-1]); ax.set_xlabel('Layer (0 = embedding)')
    ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('Structural-class probe vs layer — base vs Coder (CoST)', y=1.02)
p = os.path.join(OUTPUT_DIR, 'structure_probe_macro_f1.png')
plt.tight_layout(); plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print('saved', p)

In [ ]:
# Best-layer Python structural confusion matrix per model (computed in the driver).
names = list(ALL.keys())
fig, axes = plt.subplots(1, len(names), figsize=(6.5 * len(names), 5.5), squeeze=False)
for ax, name in zip(axes[0], names):
    cf = ALL[name]['confusion']
    cm = confusion_matrix(cf['y_true'], cf['y_pred'], labels=range(len(CLASSES)))
    cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f"{name.split('/')[-1]} — Python @ layer {cf['best_layer']}")
p = os.path.join(OUTPUT_DIR, 'structure_confusion_python.png')
plt.tight_layout(); plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print('saved', p)

In [ ]:
# Selectivity = within macro-F1 - control macro-F1 (solid) and control F1 (dashed).
# Large gap above the dashed line = representation encodes structure beyond token identity.
if any(ALL[n]['control'] for n in ALL):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), squeeze=False)
    for ax, lang in zip(axes[0], ['python', 'java']):
        for name in ALL:
            R = ALL[name]
            if not R['control']:
                continue
            L = list(range(R['N_HS']))
            real = np.array(R['within'][lang]['macro_f1'], float)
            ctrl = np.array(R['control'][lang]['macro_f1'], float)
            short = name.split('/')[-1]
            ax.plot(L, real - ctrl, '-o', ms=3, label=f'{short} (selectivity)')
            ax.plot(L, ctrl, '--', alpha=0.45, label=f'{short} (control F1)')
        ax.set_title(f'{lang}: selectivity & control')
        ax.set_xlabel('Layer'); ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    p = os.path.join(OUTPUT_DIR, 'structure_selectivity.png')
    plt.tight_layout(); plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print('saved', p)
else:
    print('control task disabled (RUN_CONTROL_TASK=False)')

In [ ]:
# Per-class within-language F1 vs layer (Python) — which class peaks at which depth.
names = list(ALL.keys())
fig, axes = plt.subplots(1, len(names), figsize=(7 * len(names), 5), squeeze=False)
for ax, name in zip(axes[0], names):
    R = ALL[name]; L = list(range(R['N_HS']))
    pcf = R['within']['python']['per_class_f1']
    for c, cname in enumerate(CLASSES):
        ax.plot(L, pcf[c], '-', label=cname)
    ax.set_title(f"{name.split('/')[-1]} — Python per-class F1")
    ax.set_xlabel('Layer'); ax.set_ylabel('F1'); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
p = os.path.join(OUTPUT_DIR, 'structure_per_class_f1.png')
plt.tight_layout(); plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print('saved', p)

In [ ]:
# Identifier-role probe (variable / function / type / parameter) — within + cross, per model.
# This is the hard probe: token identity no longer determines the label.
id_names = [n for n in ALL if ALL[n]['idrole']]
if id_names:
    fig, axes = plt.subplots(1, len(id_names), figsize=(7 * len(id_names), 5), squeeze=False)
    for ax, name in zip(axes[0], id_names):
        R = ALL[name]['idrole']; L = list(range(ALL[name]['N_HS']))
        ax.plot(L, R['within']['python']['macro_f1'], '-o', ms=3, label='Python (within)', color='#1f77b4')
        ax.plot(L, R['within']['java']['macro_f1'],   '-s', ms=3, label='Java (within)',   color='#ff7f0e')
        ax.plot(L, R['cross']['python->java']['macro_f1'], '--^', ms=3, label='Train Py → Test Java', color='#2ca02c')
        ax.plot(L, R['cross']['java->python']['macro_f1'], '--v', ms=3, label='Train Java → Test Py', color='#d62728')
        ax.axhline(R['within']['python']['baseline_f1'], color='gray', ls=':', alpha=0.7, label='Majority baseline')
        ax.set_title(f"{name.split('/')[-1]} — identifier roles")
        ax.set_xlabel('Layer (0 = embedding)'); ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    p = os.path.join(OUTPUT_DIR, 'identifier_role_macro_f1.png')
    plt.tight_layout(); plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print('saved', p)
else:
    print('identifier-role probe disabled (RUN_IDENTIFIER_ROLES=False)')

In [ ]:
# Persist numeric results for all models (NaN -> None so it's valid JSON).
def _clean(xs):
    return [None if (x is None or (isinstance(x, float) and np.isnan(x))) else float(x) for x in xs]

def _sub(a_list, b_list):
    out = []
    for a, b in zip(a_list, b_list):
        if a is None or b is None or np.isnan(a) or np.isnan(b):
            out.append(None)
        else:
            out.append(float(a - b))
    return out

def _within_block(w, class_names):
    return {lang: {'macro_f1': _clean(w[lang]['macro_f1']), 'acc': _clean(w[lang]['acc']),
                   'baseline_f1': float(w[lang]['baseline_f1']),
                   'per_class_f1': {class_names[c]: _clean(w[lang]['per_class_f1'][c])
                                    for c in range(len(class_names))}}
            for lang in w}

summary = {
    'classes': CLASSES, 'id_roles': ID_ROLES,
    'config': {'MAX_PROGRAMS_PER_LANG': MAX_PROGRAMS_PER_LANG,
               'MAX_TOKENS_PER_LANG': MAX_TOKENS_PER_LANG, 'MAX_SEQ_LEN': MAX_SEQ_LEN,
               'INCLUDE_COMMENTS': INCLUDE_COMMENTS, 'LAYER_STRIDE': LAYER_STRIDE,
               'RUN_CONTROL_TASK': RUN_CONTROL_TASK, 'RUN_IDENTIFIER_ROLES': RUN_IDENTIFIER_ROLES},
    'models': {},
}
for name, R in ALL.items():
    block = {'num_layers': R['NUM_LAYERS'],
             'within': _within_block(R['within'], CLASSES),
             'cross': {k: {'macro_f1': _clean(v['macro_f1']), 'acc': _clean(v['acc'])}
                       for k, v in R['cross'].items()}}
    if R.get('control'):
        block['control'] = {lang: {'macro_f1': _clean(R['control'][lang]['macro_f1'])}
                            for lang in R['control']}
        block['selectivity'] = {lang: _sub(R['within'][lang]['macro_f1'],
                                            R['control'][lang]['macro_f1'])
                                for lang in R['control']}
    if R.get('idrole'):
        block['identifier_roles'] = {
            'within': _within_block(R['idrole']['within'], ID_ROLES),
            'cross': {k: {'macro_f1': _clean(v['macro_f1']), 'acc': _clean(v['acc'])}
                      for k, v in R['idrole']['cross'].items()}}
    summary['models'][name] = block

p = os.path.join(OUTPUT_DIR, 'structure_probe_results.json')
with open(p, 'w') as f:
    json.dump(summary, f, indent=2)
print('saved', p, '| models:', list(summary['models'].keys()))

## Notes — how to read the comparison

- **Within-language curves:** near-ceiling for both models. High even at layer 0 because a
  token's structural class is largely its identity. The shape (early peak, slow late decline)
  is the signal, not the absolute height.
- **Cross-lingual gap:** if `Py→Java` tracks the within curve, the model encodes syntactic
  class in a shared, language-agnostic subspace. Watch whether **Coder** narrows this gap.
- **Selectivity (real − control):** the control task probes a *random fixed class per token
  type*. If control macro-F1 is also high, the probe is partly reading token identity; large
  selectivity means the representation genuinely encodes structure. Compare base vs Coder.
- **Identifier roles:** `variable/function/type/parameter` is the hard probe — token identity
  no longer suffices, so this is where code pretraining (Coder) should help most and where the
  best layer is likely *deeper* than for raw structural class.
- **Per-class curves:** show which classes peak at which depth (e.g. literals early,
  identifier roles later).
- **Scaling up:** raise `MAX_PROGRAMS_PER_LANG` / `MAX_TOKENS_PER_LANG` for tighter estimates
  (host-RAM cost ≈ `MAX_TOKENS_PER_LANG × (NUM_LAYERS+1) × hidden_size × 2 bytes` per language;
  only one model is in memory at a time). See `docs/structure_probing.md`.